字母级别的预测(预测单个字母)[已被淘汰]

In [ ]:
# 读取shakespeare.txt，构建字符级字典
with open("shakespeare.txt", "r", encoding="utf-8") as f:
    text = f.read()

# 构建字符到索引的映射字典
# set去重,排序, 转换为列表,每个字符对应一个索引
chars = sorted(list(set(text)))
#enumerate: 为每个字符分配一个唯一的索引, 从0开始
#（字符→索引）
char2idx = {ch: idx for idx, ch in enumerate(chars)}# 字符到索引的映射字典
#（索引→字符）
idx2char = {idx: ch for idx, ch in enumerate(chars)}# 索引到字符的映射字典

print(f"字符到索引的映射字典: {char2idx}")
print('-'*100)
print(f"索引到字符的映射字典: {idx2char}")
print(char2idx['!'])


字符到索引的映射字典: {'\n': 0, ' ': 1, '!': 2, '$': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '3': 9, ':': 10, ';': 11, '?': 12, 'A': 13, 'B': 14, 'C': 15, 'D': 16, 'E': 17, 'F': 18, 'G': 19, 'H': 20, 'I': 21, 'J': 22, 'K': 23, 'L': 24, 'M': 25, 'N': 26, 'O': 27, 'P': 28, 'Q': 29, 'R': 30, 'S': 31, 'T': 32, 'U': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't': 58, 'u': 59, 'v': 60, 'w': 61, 'x': 62, 'y': 63, 'z': 64}
----------------------------------------------------------------------------------------------------
索引到字符的映射字典: {0: '\n', 1: ' ', 2: '!', 3: '$', 4: '&', 5: "'", 6: ',', 7: '-', 8: '.', 9: '3', 10: ':', 11: ';', 12: '?', 13: 'A', 14: 'B', 15: 'C', 16: 'D', 17: 'E', 18: 'F', 19: 'G', 20: 'H', 21: 'I', 22: 'J', 23: 'K', 24: 'L', 25: 'M', 26: 'N', 27: 'O', 28: 'P', 29: 'Q', 30: 'R', 31: 'S', 32: 'T', 33: 'U', 

In [11]:
# 将文本转换为索引序列
# text_ids=[char2idx[ch] for ch in text]
text_ids = []
for ch in text:
    text_ids.append(char2idx[ch])
    
text_ids[0:20]

[18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64, 43, 52, 10, 0, 14, 43, 44, 53, 56]

In [13]:
print("文本长度:",len(text_ids))

文本长度: 1115394


In [16]:
import torch
from torch.utils.data import Dataset, DataLoader


# 自定义字符级数据集，用于按固定长度切分文本
class CharDataset(Dataset):
    def __init__(self, text_ids, seq_len=101):
        self.text_ids = text_ids  # 已编码的字符 id 列表
        self.seq_len = seq_len  # 每条样本的字符长度
        self.num_seq = len(text_ids) // seq_len  # 样本个数

    def __len__(self):
        return self.num_seq

    def __getitem__(self, idx):
        # 进行切片，每个样本101个字符
        return self.text_ids[idx * self.seq_len : (idx + 1) * self.seq_len]


# 创建数据集实例，假设 text_ids 已在别处定义
dataset = CharDataset(text_ids, seq_len=101)


def collate_fn(batch):
    """
    将每个样本的101个字符拆成：
    输入：前100个字符
    标签：后100个字符（每个位置对应下一个字符的预测目标）
    """
    batch = torch.tensor(batch, dtype=torch.long)
    x = batch[:, :100]  # 前100个字符作为输入
    y = batch[:, 1:101]  # 后100个字符作为标签/lable
    return x, y


# 构建 DataLoader，用于训练时按批次读取数据
dataloader = DataLoader(dataset, batch_size=64, shuffle=True, collate_fn=collate_fn)

In [19]:
for x,y in dataloader:# 遍历 dataloader，每次取出一个 batch 的数据
    print(x.shape, y.shape)
    break

torch.Size([64, 100]) torch.Size([64, 100])


# 搭建模型

In [ ]:

# 生成模型：Embedding + RNN + Linear
class CharRNNGen(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim=128, rnn_hidden=256, num_layers=2):
        super().__init__()
        self.embed = torch.nn.Embedding(vocab_size, embed_dim)      # 字符嵌入层
        self.rnn = torch.nn.RNN(embed_dim, rnn_hidden, num_layers,
                                batch_first=True, nonlinearity='tanh')  # RNN 层
        self.fc = torch.nn.Linear(rnn_hidden, vocab_size)           # 输出映射到字符类别

    def forward(self, x, h=None):
        # x: [B, T]  输入字符序列
        x = self.embed(x)                  # [B, T] -> [B, T, E]
        out, h = self.rnn(x, h)              # [B, T, H], [L, B, H]
        logits = self.fc(out)                # [B, T, V]
        return logits, h                     # 返回 logits 与隐藏状态


# 假设词汇表大小已知（示例值） ，实际使用时替换为真实 vocab_size
vocab_size = len(char2idx)   # 字符到索引的映射长度，即词汇表大小
print(f"词汇表大小: {vocab_size}")
model = CharRNNGen(vocab_size)


词汇表大小: 65


In [ ]:
# 从数据加载器取一批数据
x, y = next(iter(dataloader))

# 简单前向计算
output, hidden = model(x)
# 打印模型前向计算的输出张量形状，通常表示每个时间步的预测结果
print("output shape:", output.shape)
# 打印隐藏状态张量形状，通常包含RNN/LSTM等网络在最后一个时间步的隐层输出
print("hidden shape:", hidden.shape)

output shape: torch.Size([64, 100, 65])
hidden shape: torch.Size([2, 64, 256])


 解释张量维度含义  
 output shape: torch.Size([64, 100, 65])  
   64  -> 批次大小（batch size）  
   100 -> 序列长度（sequence length）  
   65  -> 词汇表大小（vocab size），即每个时间步输出的 logits 维度  
  
 hidden shape: torch.Size([2, 64, 256])  
   2   -> LSTM 层数（num_layers）  
   64  -> 批次大小（batch size）  
   256 -> 隐藏层维度（hidden_size）  
  
 综上：  
   output 是给定输入序列后，模型在每个时间步对所有词汇的打分；  
   hidden 是 LSTM 在最后时间步输出的隐状态，供下一步解码或继续生成使用。
